# 內部人股份質押：董監事持股

從 MOPS 追蹤董監事股份質押活動。

In [2]:
from twmops import InsidersFetcher
import pandas as pd
import asyncio

## 取得公司的質押詳情

取得董監事個別的質押紀錄。

In [3]:
def fetch_pledging_details():
    fetcher = InsidersFetcher()

    # 取得台積電 (2330) 股份質押紀錄（2026 年 3 月）
    result = fetcher.get_share_pledging("2330", year=115, month=3)

    print(f"{result.company_name} ({result.stock_id})")
    print(f"報告日期: {result.year}/{result.month}")
    print(f"總質押紀錄數: {len(result.details)}")
    print()

    # 顯示質押詳情
    for detail in result.details[:10]:
        print(f"姓名: {detail.name}")
        print(f"  職稱: {detail.title}")
        if detail.pledged_shares is not None:
            print(f"  質押股數: {detail.pledged_shares:,}")
        if detail.current_shares is not None:
            print(f"  目前持股: {detail.current_shares:,}")
        if detail.pledge_ratio is not None:
            print(f"  質押比率: {detail.pledge_ratio:.2%}")
        print()

    if len(result.details) > 10:
        print(f"... 還有 {len(result.details) - 10} 筆紀錄")

    return result

pledging_response = fetch_pledging_details()

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


台灣積體電路製造股份有限公司 (2330)
Report date: 115/3
Total pledging records: 41

Person: 魏哲家
  Title: 董事長
  Pledged shares: 1,600,000
  Current shares: 7,452,349
  Pledge ratio: 2146.00%

Person: 行政院國家發展基金管理會
  Title: 董事
  Pledged shares: 0
  Current shares: 1,653,709,980
  Pledge ratio: 0.00%

Person: 葉俊顯
  Title: 董事之法人代表人
  Pledged shares: 0
  Current shares: 0
  Pledge ratio: 0.00%

Person: 曾繁城
  Title: 董事
  Pledged shares: 0
  Current shares: 29,472,675
  Pledge ratio: 0.00%

Person: 彼得‧邦菲爵士
  Title: 獨立董事
  Pledged shares: 0
  Current shares: 0
  Pledge ratio: 0.00%

Person: 拉斐爾．萊夫
  Title: 獨立董事
  Pledged shares: 0
  Current shares: 0
  Pledge ratio: 0.00%

Person: 麥克‧史賓林特
  Title: 獨立董事
  Pledged shares: 0
  Current shares: 0
  Pledge ratio: 0.00%

Person: 摩西．蓋弗瑞洛夫
  Title: 獨立董事
  Pledged shares: 0
  Current shares: 0
  Pledge ratio: 0.00%

Person: 烏蘇拉‧伯恩斯
  Title: 獨立董事
  Pledged shares: 0
  Current shares: 0
  Pledge ratio: 0.00%

Person: 琳恩‧埃爾森漢斯
  Title: 獨立董事
  Pledged shares: 0
  Current s

## 分析質押摘要

按職位分類質押統計。

In [4]:
def analyze_pledging_summary():
    fetcher = InsidersFetcher()

    result = fetcher.get_share_pledging("2330", year=115, month=3)

    print(f"{result.company_name} 質押摘要")
    print(f"報告日期: {result.year}/{result.month}")
    print()

    # 按職位分類
    df = pd.DataFrame([
        {
            'name': detail.name,
            'title': detail.title,
            'current_shares': detail.current_shares,
            'pledged_shares': detail.pledged_shares,
            'pledge_ratio': detail.pledge_ratio,
        }
        for detail in result.details
    ])

    # 按職位摘要
    print("按職位分類摘要:")
    summary = df.groupby('title').agg({
        'current_shares': 'sum',
        'pledged_shares': 'sum',
        'pledge_ratio': 'mean',
    }).round(4)
    summary['count'] = df.groupby('title').size()
    print(summary)
    print()

    # 質押最多的人（篩除 None 值）
    print("質押比率前 5 高:")
    df_with_ratio = df[df['pledge_ratio'].notna()]
    if len(df_with_ratio) > 0:
        top_pledgers = df_with_ratio.nlargest(5, 'pledge_ratio')[['name', 'title', 'pledged_shares', 'current_shares', 'pledge_ratio']]
        print(top_pledgers.to_string(index=False))
    else:
        print("無質押比率資料")

    return df

pledging_df = analyze_pledging_summary()

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


Pledging Summary for 台灣積體電路製造股份有限公司
Report date: 115/3

Summary by Title:
          current_shares  pledged_shares  pledge_ratio  count
title                                                        
副總經理            24291094         1128000        2.0919     27
會計部門主管              5367               0        0.0000      1
獨立董事              126826               0        0.0000      7
總經理              7452349         1600000       21.4600      1
董事            1683182655               0        0.0000      2
董事之法人代表人               0               0        0.0000      1
董事長              7452349         1600000       21.4600      1
財務部門主管           1811543               0        0.0000      1

Top 5 Pledgers (by pledge_ratio):
name title  pledged_shares  current_shares  pledge_ratio
  何軍  副總經理           28000          120119         23.31
 何麗梅  副總經理         1000000         4644012         21.53
 魏哲家   董事長         1600000         7452349         21.46
 魏哲家   總經理         1600000         7452349 

## 欄位參考

每筆質押紀錄包含：
- `name`: 董監事姓名
- `title`: 職位（董事長、董事、監察人等）
- `current_shares`: 目前持股數
- `pledged_shares`: 質押股數
- `pledge_ratio`: 質押股數 / 目前持股數